# Popularidade vs. Qualidade

Esta análise mostra se livros famosos (com muitas avaliações) são realmente os melhores avaliados.

In [6]:
import pandas as pd
import plotly.express as px

# Carregando os dados limpos gerados no Notebook 00
try:
    df = pd.read_csv('datasets/books.csv')
    print("Dataset carregado com sucesso!")
except FileNotFoundError:
    print("Erro: Arquivo 'datasets/books.csv' não encontrado. Certifique-se de rodar o Notebook 00 primeiro.")

# Dispersão entre Popularidade (ratings_count) e Qualidade (average_rating)
# Filtramos livros com pelo menos 1 avaliação para evitar ruído
df_pop = df[df['ratings_count'] > 0].copy()

fig1 = px.scatter(
    df_pop, 
    x='average_rating', 
    y='ratings_count', 
    log_y=True, 
    hover_name='title',
    hover_data=['authors', 'published_year'],
    title='Popularidade (Qtd Avaliações) vs. Qualidade (Nota Média)',
    labels={'average_rating': 'Nota Média', 'ratings_count': 'Total de Avaliações (Escala Log)'},
    color='average_rating',
    color_continuous_scale='Reds',
    template='plotly_white'
)

fig1.show()

Dataset carregado com sucesso!


# Top Autores de Elite

Filtramos autores que possuem um volume considerável de obras (mínimo 5) e calculamos quem tem a maior média de notas.

In [2]:
# Autores com maior consistência de qualidade
contagem_autores = df['authors'].value_counts()
# Consideramos apenas autores com 5 ou mais livros no dataset
autores_elite_lista = contagem_autores[contagem_autores >= 5].index
df_elite = df[df['authors'].isin(autores_elite_lista)]

author_stats = df_elite.groupby('authors').agg(
    Media_Nota=('average_rating', 'mean'),
    Qtd_Livros=('title', 'count')
).reset_index().sort_values(by='Media_Nota', ascending=False).head(10)

fig2 = px.bar(
    author_stats, 
    x='Media_Nota', 
    y='authors', 
    orientation='h',
    title='Top 10 Autores com Melhor Média de Notas (Mín. 5 livros)',
    labels={'Media_Nota': 'Nota Média Geral', 'authors': 'Autor'},
    color='Media_Nota',
    color_continuous_scale='YlOrRd',
    hover_data=['Qtd_Livros']
)

# Ajustando o eixo X para destacar as diferenças nas notas altas
fig2.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_range=[3.5, 5])
fig2.show()

# Evolução das Notas por Década

Analisamos se a percepção de qualidade mudou ao longo do tempo.

In [9]:
# Linha do tempo das notas médias por década
# Filtramos anos realistas (entre 1800 e 2024)
df_climatizado = df[(df['published_year'] >= 1800) & (df['published_year'] <= 2024)].copy()
df_climatizado['Decada'] = (df_climatizado['published_year'] // 10) * 10

decada_stats = df_climatizado.groupby('Decada')['average_rating'].mean().reset_index()

fig3 = px.line(
    decada_stats, 
    x='Decada', 
    y='average_rating', 
    markers=True,
    title='Tendência Histórica: Nota Média de Livros por Década',
    labels={'Decada': 'Década de Publicação', 'average_rating': 'Média das Notas'},
    color_discrete_sequence=['#6F1D1B']
)

fig3.update_layout(hovermode="x unified")
fig3.show()

# Conclusão: Padrões de Qualidade e Engajamento

Neste notebook, exploramos variáveis fundamentais para a construção de um sistema de recomendação robusto para o aplicativo Booklog. As análises interativas nos permitiram extrair três conclusões principais:

### 1. O Paradoxo da Popularidade vs. Qualidade: 
Observamos que os livros com as maiores notas absolutas (próximas a 5.0) geralmente pertencem à "cauda longa", ou seja, são obras de nicho com pouquíssimas avaliações. Em contrapartida, os blockbusters (com milhões de avaliações) tendem a ter suas notas estabilizadas em uma média mais realista (entre 3.8 e 4.5). Impacto no App: Nosso futuro algoritmo não poderá recomendar livros baseando-se apenas na nota máxima bruta, pois acabaria indicando obras desconhecidas. Será necessário criar um "peso" que equilibre a nota com a quantidade de leitores (como o cálculo Bayesiano).

### 2. A Consistência dos Autores de Elite: 
Identificamos que existe um grupo seleto de autores que, mesmo com um volume alto de publicações (5+ livros), conseguem manter médias de avaliação altíssimas. Impacto no App: Estes autores representam o que chamamos de "Apostas Seguras" (Safe Bets). Eles serão excelentes candidatos para solucionar o problema de Cold Start (quando um usuário novo entra no app e ainda não sabemos o que ele gosta, recomendamos esses mestres da literatura).

### 3.A Percepção Histórica da Qualidade: 
A análise temporal demonstrou flutuações nas notas médias dependendo da década de publicação da obra, revelando possíveis vieses de nostalgia (clássicos sendo supervalorizados por seu status) ou tendências de consumo moderno. Impacto no App: O ano de publicação (published_year) provou ser uma feature (variável) valiosa que deverá ser incluída no treinamento do nosso modelo de Machine Learning para agrupar leitores com preferências temporais similares (ex: leitores de clássicos vs. leitores de lançamentos).